In [1]:
import numpy as np
from collections import Counter
from itertools import combinations, permutations
from ase.build import surface, make_supercell
from ase.io import write, read
from ase.visualize import view
from ase import Atom
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.io.lammps.data import LammpsData
from pymatgen.core.surface import Slab
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
import os

os.makedirs("slabs", exist_ok=True)

ld = LammpsData.from_file('bulk_structure/mg21zn25_relaxed.data', atom_style='atomic')
alloy = AseAtomsAdaptor().get_atoms(ld.structure)
print(f"Bulk: {len(alloy)} atoms, {alloy.get_chemical_formula()}")

Bulk: 276 atoms, Mg126Zn150


In [2]:
slab = surface(alloy, (1, 0, 0), 2, vacuum=8)

sym = np.array(slab.get_chemical_symbols())
mg, zn = sum(sym == 'Mg'), sum(sym == 'Zn')
z = slab.get_positions()[:, 2]
thick = z.max() - z.min()

ps = AseAtomsAdaptor().get_structure(slab)
slab_obj = Slab(ps.lattice, ps.species, ps.frac_coords,
    miller_index=(1,0,0), oriented_unit_cell=ps, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

print(f"Atoms: {len(slab)}, Mg: {mg}, Zn: {zn}")
print(f"Thickness: {thick:.1f} A")
print(f"Ratio Zn/Mg: {zn/mg:.4f} (need {25/21:.4f})")
print(f"Stoichiometric: {'YES' if zn*21 == mg*25 else 'NO'}")
print(f"Symmetric: {slab_obj.is_symmetric()}")

Atoms: 552, Mg: 252, Zn: 300
Thickness: 45.2 A
Ratio Zn/Mg: 1.1905 (need 1.1905)
Stoichiometric: YES
Symmetric: False


In [3]:
z = slab.get_positions()[:, 2]
sym = np.array(slab.get_chemical_symbols())
order = np.argsort(z)

z_sorted = np.sort(z)
gaps = np.diff(z_sorted)
tol = max(0.02, min(0.3, np.median(gaps[gaps > 0.01]) / 2))
print(f"Using tolerance: {tol:.3f} A\n")

layers = []
layer_idx = []
cur = [order[0]]
for i in range(1, len(order)):
    if z[order[i]] - z[order[i-1]] < tol:
        cur.append(order[i])
    else:
        layers.append(Counter(sym[j] for j in cur))
        layer_idx.append(list(cur))
        cur = [order[i]]
layers.append(Counter(sym[j] for j in cur))
layer_idx.append(list(cur))

z_min, z_max = z.min(), z.max()
n = len(layers)

print(f"Total atomic layers: {n}\n")
print(f"{'Bot':>7} {'Bot Comp':>10} {'from_bot':>9}  |  {'Top':>7} {'Top Comp':>10} {'from_top':>9} {'Match':>6}")
print("-" * 80)
for i in range(min(15, n//2)):
    bot, top = layers[i], layers[n-1-i]
    zm_b = np.mean([z[j] for j in layer_idx[i]])
    zm_t = np.mean([z[j] for j in layer_idx[n-1-i]])
    
    mg_b, zn_b = bot.get('Mg',0), bot.get('Zn',0)
    mg_t, zn_t = top.get('Mg',0), top.get('Zn',0)
    comp_b = f"Mg{mg_b}Zn{zn_b}" if mg_b and zn_b else (f"Mg{mg_b}" if mg_b else f"Zn{zn_b}")
    comp_t = f"Mg{mg_t}Zn{zn_t}" if mg_t and zn_t else (f"Mg{mg_t}" if mg_t else f"Zn{zn_t}")
    
    match = "YES" if bot == top else "NO <---"
    print(f"{i:>7} {comp_b:>10} {zm_b-z_min:>9.3f}  |  {n-1-i:>7} {comp_t:>10} {z_max-zm_t:>9.3f} {match:>6}")

Using tolerance: 0.039 A

Total atomic layers: 175

    Bot   Bot Comp  from_bot  |      Top   Top Comp  from_top  Match
--------------------------------------------------------------------------------
      0     Mg4Zn6     0.000  |      174        Mg2     0.000 NO <---
      1        Mg2     0.041  |      173        Mg2     0.040    YES
      2        Mg2     0.420  |      172        Mg2     0.420    YES
      3        Mg2     0.619  |      171        Mg2     0.618    YES
      4        Zn4     1.320  |      170        Zn4     1.319    YES
      5        Zn2     1.364  |      169        Zn2     1.364    YES
      6        Zn2     2.202  |      168        Zn2     2.201    YES
      7     Mg2Zn2     2.269  |      167     Mg2Zn2     2.269    YES
      8        Mg2     2.329  |      166        Mg2     2.328    YES
      9        Mg2     2.444  |      165        Mg2     2.443    YES
     10        Zn2     2.522  |      164        Zn2     2.521    YES
     11     Mg4Zn6     2.663  |      1

In [4]:
# Search for removals that give symmetry
print("Searching for symmetric removals...\n")

for bot_rm in range(0, 6):
    for top_rm in range(0, 6):
        if bot_rm == 0 and top_rm == 0:
            continue
        keep = []
        for j in range(bot_rm, n - top_rm):
            keep.extend(layer_idx[j])
        if len(keep) == 0:
            continue
        tr = slab[keep]
        try:
            ps_tr = AseAtomsAdaptor().get_structure(tr)
            slab_tr = Slab(ps_tr.lattice, ps_tr.species, ps_tr.frac_coords,
                miller_index=(1,0,0), oriented_unit_cell=ps_tr, shift=0,
                scale_factor=[[1,0,0],[0,1,0],[0,0,1]])
            if slab_tr.is_symmetric():
                removed_bot = sum(len(layer_idx[j]) for j in range(bot_rm))
                removed_top = sum(len(layer_idx[j]) for j in range(n - top_rm, n))
                sym_tr = np.array(tr.get_chemical_symbols())
                mg_tr = sum(sym_tr == 'Mg')
                zn_tr = sum(sym_tr == 'Zn')
                print(f"  bot_rm={bot_rm}, top_rm={top_rm}: {len(tr)} atoms, "
                      f"Mg{mg_tr} Zn{zn_tr}, removed {removed_bot}+{removed_top}={removed_bot+removed_top}")
        except:
            continue

Searching for symmetric removals...

  bot_rm=1, top_rm=1: 540 atoms, Mg246 Zn294, removed 10+2=12
  bot_rm=2, top_rm=2: 536 atoms, Mg242 Zn294, removed 12+4=16
  bot_rm=3, top_rm=3: 532 atoms, Mg238 Zn294, removed 14+6=20
  bot_rm=4, top_rm=4: 528 atoms, Mg234 Zn294, removed 16+8=24
  bot_rm=5, top_rm=5: 520 atoms, Mg234 Zn286, removed 20+12=32


In [5]:
# Remove 1 layer from each end
removed_bot = layer_idx[0]
removed_top = layer_idx[n-1]
removed_all = removed_bot + removed_top

keep = []
for j in range(1, n - 1):
    keep.extend(layer_idx[j])

trimmed = slab[keep]
ps_trim = AseAtomsAdaptor().get_structure(trimmed)

sga = SpacegroupAnalyzer(ps_trim, symprec=0.1)
print(f"SG = {sga.get_space_group_symbol()}")

for op in sga.get_symmetry_operations():
    if op.rotation_matrix[2][2] < 0:
        inv_translation = op.translation_vector
        print(f"Inversion: f -> {inv_translation} - f")
        break

rem_mg = sum(sym[j] == 'Mg' for j in removed_all)
rem_zn = sum(sym[j] == 'Zn' for j in removed_all)
print(f"\nRemoved: {len(removed_all)} atoms (Mg{rem_mg} Zn{rem_zn})")

print(f"\nBottom layer ({len(removed_bot)}):")
for j in removed_bot:
    pos = slab.positions[j]
    print(f"  idx={j} {sym[j]} ({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f})")

print(f"\nTop layer ({len(removed_top)}):")
for j in removed_top:
    pos = slab.positions[j]
    print(f"  idx={j} {sym[j]} ({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f})")

# Check partners
ps_orig = AseAtomsAdaptor().get_structure(slab)
keep_set = set(keep)

paired = []
unpaired = []
for j in removed_all:
    frac = ps_orig[j].frac_coords
    inv_frac = (inv_translation - frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    
    # Check against other removed atoms
    matched = False
    for k in removed_all:
        if k == j:
            continue
        if np.linalg.norm(slab.positions[k] - inv_cart) < 0.5:
            paired.append((j, k))
            matched = True
            break
    if not matched:
        unpaired.append(j)
        print(f"  idx={j} {sym[j]} -> UNPAIRED, inv at ({inv_cart[0]:.3f}, {inv_cart[1]:.3f}, {inv_cart[2]:.3f})")

print(f"\nPaired within removed: {len(paired)//2} pairs")
print(f"Unpaired: {len(unpaired)}")
for j in unpaired:
    pos = slab.positions[j]
    print(f"  idx={j} {sym[j]} ({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f})")

SG = P2/c
Inversion: f -> [0. 0. 0.] - f

Removed: 12 atoms (Mg6 Zn6)

Bottom layer (10):
  idx=85 Mg (18.129, 2.197, 8.000)
  idx=222 Zn (14.502, 6.591, 8.000)
  idx=275 Zn (24.545, 6.591, 8.000)
  idx=2 Zn (0.000, 0.000, 8.000)
  idx=139 Zn (0.000, 4.394, 8.000)
  idx=1 Zn (1.575, 2.197, 8.000)
  idx=84 Zn (11.619, 2.197, 8.000)
  idx=166 Mg (7.992, 6.591, 8.000)
  idx=137 Mg (23.078, 3.830, 8.001)
  idx=161 Mg (3.043, 8.224, 8.001)

Top layer (2):
  idx=391 Mg (23.078, 0.564, 53.242)
  idx=414 Mg (3.043, 4.958, 53.242)
  idx=85 Mg -> UNPAIRED, inv at (7.992, 6.591, 53.242)
  idx=222 Zn -> UNPAIRED, inv at (11.619, 2.197, 53.242)
  idx=275 Zn -> UNPAIRED, inv at (1.575, 2.197, 53.242)
  idx=2 Zn -> UNPAIRED, inv at (26.121, 0.000, 53.242)
  idx=139 Zn -> UNPAIRED, inv at (26.121, 4.394, 53.242)
  idx=1 Zn -> UNPAIRED, inv at (24.545, 6.591, 53.242)
  idx=84 Zn -> UNPAIRED, inv at (14.502, 6.591, 53.242)
  idx=166 Mg -> UNPAIRED, inv at (18.129, 2.197, 53.242)

Paired within removed: 

In [6]:
# 3 mutual pairs among the 8 unpaired: (85,166), (222,84), (275,1)
# Plus idx=2 and idx=139 — check if they're a pair or self-inverses

ps_orig = AseAtomsAdaptor().get_structure(slab)

# Check idx=2 and idx=139
for j in [2, 139]:
    frac = ps_orig[j].frac_coords
    inv_frac = (-frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    d_to_2 = np.linalg.norm(slab.positions[2] - inv_cart)
    d_to_139 = np.linalg.norm(slab.positions[139] - inv_cart)
    print(f"idx={j}: inv at ({inv_cart[0]:.3f}, {inv_cart[1]:.3f}, {inv_cart[2]:.3f}), "
          f"dist to idx=2: {d_to_2:.3f}, dist to idx=139: {d_to_139:.3f}")

# Reconstruct: keep one from each pair on bottom, move other to top (inv position)
# Pairs: (85,166), (222,84), (275,1), and (2,139) if they're a pair
sc_fixed = slab.copy()

move_pairs = [
    (166, 85),    # move 166 to inv(85) — Mg
    (84, 222),    # move 84 to inv(222) — Zn
    (1, 275),     # move 1 to inv(275) — Zn
    (139, 2),     # move 139 to inv(2) — Zn
]

for move_idx, keep_idx in move_pairs:
    frac = ps_orig[keep_idx].frac_coords
    inv_frac = (-frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    old = sc_fixed.positions[move_idx].copy()
    sc_fixed.positions[move_idx] = inv_cart
    print(f"Moved {move_idx} ({sym[move_idx]}) -> inv({keep_idx}): "
          f"({old[0]:.3f}, {old[1]:.3f}, {old[2]:.3f}) -> "
          f"({inv_cart[0]:.3f}, {inv_cart[1]:.3f}, {inv_cart[2]:.3f})")

sym_f = np.array(sc_fixed.get_chemical_symbols())
mg_f = sum(sym_f == 'Mg')
zn_f = sum(sym_f == 'Zn')

ps_fixed = AseAtomsAdaptor().get_structure(sc_fixed)
slab_fixed = Slab(ps_fixed.lattice, ps_fixed.species, ps_fixed.frac_coords,
    miller_index=(1,0,0), oriented_unit_cell=ps_fixed, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

print(f"\nAtoms: {len(sc_fixed)}, Mg: {mg_f}, Zn: {zn_f}")
print(f"Stoichiometric: {'YES' if zn_f*21 == mg_f*25 else 'NO'}")
print(f"Symmetric: {slab_fixed.is_symmetric()}")

idx=2: inv at (26.121, 0.000, 53.242), dist to idx=2: 52.241, dist to idx=139: 52.426
idx=139: inv at (26.121, 4.394, 53.242), dist to idx=2: 52.426, dist to idx=139: 52.241
Moved 166 (Mg) -> inv(85): (7.992, 6.591, 8.000) -> (7.992, 6.591, 53.242)
Moved 84 (Zn) -> inv(222): (11.619, 2.197, 8.000) -> (11.619, 2.197, 53.242)
Moved 1 (Zn) -> inv(275): (1.575, 2.197, 8.000) -> (1.575, 2.197, 53.242)
Moved 139 (Zn) -> inv(2): (0.000, 4.394, 8.000) -> (26.121, 0.000, 53.242)

Atoms: 552, Mg: 252, Zn: 300
Stoichiometric: YES
Symmetric: True


In [7]:
view(sc_fixed)

<Popen: returncode: None args: ['C:\\Users\\Kai Savage\\anaconda3\\python.ex...>

In [8]:
thick = sc_fixed.get_positions()[:,2].max() - sc_fixed.get_positions()[:,2].min()
area = np.linalg.norm(np.cross(sc_fixed.cell[0], sc_fixed.cell[1]))

ps = AseAtomsAdaptor().get_structure(sc_fixed)
ld = LammpsData.from_structure(ps, atom_style="atomic")
ld.write_file("slabs/mg21zn25_100_2layers_reconstructed_ase.data")

print(f"Saved: slabs/mg21zn25_100_2layers_reconstructed_ase.data")
print(f"  Atoms: {len(sc_fixed)}")
print(f"  Thickness: {thick:.1f} A")
print(f"  Area: {area:.1f} A²")
print(f"  Stoichiometric: YES")
print(f"  Symmetric: TRUE")

Saved: slabs/mg21zn25_100_2layers_reconstructed_ase.data
  Atoms: 552
  Thickness: 45.2 A
  Area: 229.5 A²
  Stoichiometric: YES
  Symmetric: TRUE


In [9]:
#unrelaxed surface energy calculation
E_slab = -705.08616         # eV
E_bulk_per_fu = -363.4709  / 6  # eV per formula unit = 
n = 552 / 46                # formula units in slab =
A = 229.5           # Ang^2

S_eV_per_ang2 = (E_slab - n * E_bulk_per_fu) / (2 * A)

# Convert to J/m^2 (1 eV/Ang^2 = 16.0218 J/m^2)
S_J_per_m2 = S_eV_per_ang2 * 16.0218

print(f"E_bulk per formula unit = {E_bulk_per_fu:.6f} eV")
print(f"n * E_bulk = {n * E_bulk_per_fu:.5f} eV")
print(f"E_slab - n*E_bulk = {E_slab - n * E_bulk_per_fu:.5f} eV")
print(f"S = {S_eV_per_ang2:.6f} eV/Ang^2")
print(f"S = {S_J_per_m2:.4f} J/m^2")

E_bulk per formula unit = -60.578483 eV
n * E_bulk = -726.94180 eV
E_slab - n*E_bulk = 21.85564 eV
S = 0.047616 eV/Ang^2
S = 0.7629 J/m^2


In [10]:
#relaxed surface energy calculation
E_slab_relaxed = -708.459556788316   # eV
E_bulk_per_fu =  -363.4709  / 6  # eV per formula unit
n = 552 / 46                      # 32 formula units
A = 229.5                 # Ang^2

S_relaxed_eV = (E_slab_relaxed - n * E_bulk_per_fu) / (2 * A)
S_relaxed_Jm2 = S_relaxed_eV * 16.0218

print(f"E_slab_relaxed - n*E_bulk = {E_slab_relaxed - n * E_bulk_per_fu:.5f} eV")
print(f"S relaxed = {S_relaxed_eV:.6f} eV/Ang^2")
print(f"S relaxed = {S_relaxed_Jm2:.4f} J/m^2")

# Compare with unrelaxed
S_unrelaxed_Jm2 = 0.7629
print(f"\nUnrelaxed S = {S_unrelaxed_Jm2:.4f} J/m^2")
print(f"Relaxed S   = {S_relaxed_Jm2:.4f} J/m^2")
print(f"Relaxation energy = {S_unrelaxed_Jm2 - S_relaxed_Jm2:.4f} J/m^2")

E_slab_relaxed - n*E_bulk = 18.48224 eV
S relaxed = 0.040266 eV/Ang^2
S relaxed = 0.6451 J/m^2

Unrelaxed S = 0.7629 J/m^2
Relaxed S   = 0.6451 J/m^2
Relaxation energy = 0.1178 J/m^2
